In [ ]:
import pandas as pd
import entsoe
import cdsapi
from dotenv import load_dotenv
import os
import time
import calendar


_ = load_dotenv()

# Data Acquisition
In our first step we need to gather the data we will be working with.
Please execute the cell above this. 
It will import the needed packages and load your API keys into your environment.

----

We will be using two major data sources: the ENTSO-E Transparency Platform [1] for all data regarding the energy markets and the ERA5/Copernicus Dataset [2] for weather data. These datasets are quite extensive, accurate and easily acquired.
We will download the data and store them in a file to be processed and analyzed in later steps.

## ENTSO-E

We are starting with the ENTSO-E Datasets as they are the ones we are primarily trying to analyze.
Since I am based in germany we will only be using the data of Germany and since I want to consider the effect of renewable energy I will also include Denmark.
However the principals laid out in this project should be adaptable to most other european countries.
We will only be using data from the years 2019 until 2025. This includes major market disruptions due to the ukraine war and the COVID-19 Pandemic.
The Datasets we will be using are:
  - Day-Ahead Prices
  - Actual Total Load
  - Aggregated Generation per Type

The reason to choose these is that energy prices are heavily influenced by the _Merit-Order-Effect_ [3].
This model orders the different generators from cheapest running cost to highest running cost.
That is why the Aggregated Generation per Type is interesting to us.
The model then checks what the cheapest set of generators are which will still cover the demand.
That is why the Actual Total Load is interesting.
Finally the price of energy is determined by the running cost of the most expensive generator needed to cover demand.
All other generators are able to sell their 'cheaper' energy at the more expensive price.

In [ ]:
OUTPUT_DIR = 'data/raw/entsoe'
YEARS = range(2019,2025+1)
COUNTRIES = ['DE_LU', 'DK1', 'DK2']


os.makedirs(OUTPUT_DIR, exist_ok=True)
client = entsoe.EntsoePandasClient(api_key=os.environ['ENTSOE_API_KEY'])

for country in COUNTRIES:
    for year in YEARS:
        start = pd.Timestamp(f'{year}0101', tz='UTC')
        end = pd.Timestamp(f'{year}1231', tz='UTC')
        country_code = 'DE_LU'     #  The bidding zone for Germany is the same as for Luxembourg
        target = os.path.join(OUTPUT_DIR, f'{country}_Price_{year}.parquet')
        if os.path.exists(target):
            print(f'skipping {target} (already exists)')
        else:
            print(f'requesting: {country}, {year}, price -> {target}')
            df = client.query_day_ahead_prices(country_code, start, end).to_frame(name="price")
            print('received')
            df.to_parquet(target)
        target = os.path.join(OUTPUT_DIR, f'{country}_Generation_{year}.parquet')
        if os.path.exists(target):
            print(f'skipping {target} (already exists)')
        else:
            print(f'requesting: {country}, {year}, generation -> {target}')
            df = client.query_generation(country_code=country_code, start=start, end=end, nett=False)
            print('received')
            df.to_parquet(target)
        target = os.path.join(OUTPUT_DIR, f'{country}_Load_{year}.parquet')
        if os.path.exists(target):
            print(f'skipping {target} (already exists)')
        else:
            print(f'requesting: {country}, {year}, load -> {target}')
            df = client.query_load(country_code=country_code, start=start, end=end)
            print('received')
            df.to_parquet(target)

## Copernicus
Weather data is hugely important for the energy markets,
it directly influences both sides of the Merit-Order-Model: the generation capacity of wind, solar and water energy directly correspond to the weather you are having (or had).
But also the energy consumption changes dramatically with the weather. If it is cold people will be consuming more energy to heat.
While on particularly hot days people might be more prone to turning on air conditioning.

In particularly extreme cases weather can even produce outages and disrupt the entire energy network.

_Note: I am unsure of how much industrial energy consumption varies with the weather._

In [ ]:
OUTPUT_DIR = 'data/raw/era5'
DATASET = 'reanalysis-era5-single-levels'

# rough bounding boxes
COUNTRY_AREAS = { # [North, West, South, East]
    'germany': [55.1, 5.8, 47.2, 15.1],
    'luxembourg': [50.2, 5.7, 49.4, 6.5],
    'denmark': [57.8, 8.0, 54.5, 15.2],
}

VARIABLES = [
    '10m_u_component_of_wind',
    '10m_v_component_of_wind',
    '100m_u_component_of_wind',
    '100m_v_component_of_wind',
    '2m_temperature',
    'surface_solar_radiation_downwards',
]

YEARS = range(2019,2025+1)
MONTHS = range(1, 13)
ALL_HOURS = [f'{h:02d}:00' for h in range(24)]


def build_request(area: list[float], year: int, month: int) -> dict:
    days_per_month = {
        month: [f'{d:02d}' for d in range(1, calendar.monthrange(year, month)[1] + 1)]
        for month in range(1, 13)
    }
    all_days = [f'{d:02d}' for d in range(1, 32)]

    return {
        'product_type': ['reanalysis'],
        'variable': VARIABLES,
        'year': [str(year)],
        'month': [f'{month:02d}'],
        'day': all_days,
        'time': ALL_HOURS,
        'area': area,  # [North, West, South, East]
        'data_format': 'netcdf',
        'download_format': 'unarchived',
    }


def download_all() -> None:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    client = cdsapi.Client(url=os.environ['CDS_API_URL'], key= os.environ['CDS_API_KEY'])

    for country, area in COUNTRY_AREAS.items():
        for year in YEARS:
            for month in MONTHS:
                target = os.path.join(OUTPUT_DIR, f'era5_{country}_{year}_{month}.nc')
                if os.path.exists(target):
                    print(f'skipping {target} (already exists)')
                    continue
    
                request = build_request(area, year, month)
                print(f'requesting: {country}, {year}, {month} -> {target}')
                client.retrieve(DATASET, request, target)


download_all()


requesting: germany, 2019, 1 -> data/raw/era5/era5_germany_2019_1.nc


2026-09-08 23:31:28,937 INFO Request ID is 7f23a776-e612-4d0e-94ae-dc69071e6627
2026-09-08 23:31:29,006 INFO status has been updated to accepted
2026-09-08 23:32:03,573 INFO status has been updated to successful


skipping data/raw/era5/era5_germany_2019_2.nc (already exists)
skipping data/raw/era5/era5_germany_2019_3.nc (already exists)
skipping data/raw/era5/era5_germany_2019_4.nc (already exists)
skipping data/raw/era5/era5_germany_2019_5.nc (already exists)
skipping data/raw/era5/era5_germany_2019_6.nc (already exists)
skipping data/raw/era5/era5_germany_2019_7.nc (already exists)
skipping data/raw/era5/era5_germany_2019_8.nc (already exists)
skipping data/raw/era5/era5_germany_2019_9.nc (already exists)
skipping data/raw/era5/era5_germany_2019_10.nc (already exists)
skipping data/raw/era5/era5_germany_2019_11.nc (already exists)
skipping data/raw/era5/era5_germany_2019_12.nc (already exists)
skipping data/raw/era5/era5_germany_2020_1.nc (already exists)
skipping data/raw/era5/era5_germany_2020_2.nc (already exists)
skipping data/raw/era5/era5_germany_2020_3.nc (already exists)
skipping data/raw/era5/era5_germany_2020_4.nc (already exists)
skipping data/raw/era5/era5_germany_2020_5.nc (alrea

2026-09-08 23:32:06,379 INFO Request ID is 2be2185a-ac51-4d05-aa27-574c5691681d
2026-09-08 23:32:06,439 INFO status has been updated to accepted
2026-09-08 23:32:58,156 INFO status has been updated to successful


requesting: germany, 2020, 11 -> data/raw/era5/era5_germany_2020_11.nc


...

2026-09-09 12:01:24,045 INFO Request ID is a9fa3210-a950-4a16-acbe-64066c76e65f
2026-09-09 12:01:24,167 INFO status has been updated to accepted
2026-09-09 13:40:48,861 INFO status has been updated to running
2026-09-09 13:43:10,392 INFO status has been updated to successful


requesting: luxembourg, 2020, 10 -> data/raw/era5/era5_luxembourg_2020_10.nc


2026-09-09 13:43:12,385 INFO Request ID is feab4663-d3f8-42ec-9b0b-6b4383bfbcdc
2026-09-09 13:43:12,466 INFO status has been updated to accepted
2026-09-09 14:57:59,347 INFO status has been updated to running
2026-09-09 15:02:01,111 INFO status has been updated to successful


requesting: luxembourg, 2020, 11 -> data/raw/era5/era5_luxembourg_2020_11.nc


2026-09-09 15:02:03,931 INFO Request ID is 697b939f-d868-4a8e-b803-17a6a5604a9e
2026-09-09 15:02:04,104 INFO status has been updated to accepted


## Sources
[1] ENTSO-E, "ENTSO-E Transparency Platform", 2026. [Online]. Available: https://transparency.entsoe.eu/. [Accessed: 07.09.2026].
  
[2] H. Hersbach et al., "ERA5 hourly data on single levels from 1940 to present",
    Copernicus Climate Change Service (C3S) Climate Data Store (CDS), 2018.
    [Online]. Available: https://doi.org/10.24381/cds.adbb2d47.
    [Accessed: 08.09.2026].
    
[3] F. Sensfuß, M. Ragwitz, and M. Genoese, "The merit-order effect: A detailed
    analysis of the price effect of renewable electricity generation on spot
    market prices in Germany", Energy Policy, vol. 36, no. 8, pp. 3076–3084, 2008.